# 02: Network Metrics Analysis

This notebook computes four key network metrics for each domestic flight
network:

1. **Weighted degree rank** shows whether traffic is concentrated in a few airports.
2. **Degree vs. betweenness centrality** helps identify transfer hubs and bridge airports.
3. **Degree assortativity** helps compare hub and spoke structure with more distributed networks.
4. **k+ core profile** estimates core size and network robustness.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
from src.data_preprocessing import load_data, correct_coordinates
from src.network_construction import build_graph
from src.network_metrics import (
    weighted_degree_rank, degree_betweenness, assortativity, kplus_core
)
from src.visualisation import (
    plot_weighted_degree, plot_degree_betweenness,
    plot_assortativity, plot_kplus_core
)

In [ ]:
airports, flights = load_data('../data/Airports.csv', '../data/Flight Data.xlsx')
airports = correct_coordinates(airports)

countries = ['USA', 'China', 'United Kingdom', 'Australia']
graphs = {c: build_graph(flights, c) for c in countries}

for c, G in graphs.items():
    print(f'{c}: {len(G.nodes)} nodes, {len(G.edges)} edges')

## Weighted degree rank distribution

Weighted degree (k_w) is the sum of edge weights connected to a node.
A steep drop indicates traffic concentration in a few dominant hubs, known in the literature as the King Pauper effect.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, country in enumerate(countries):
    ax = axes[idx // 2, idx % 2]
    ranks, sorted_kw = weighted_degree_rank(graphs[country])
    plot_weighted_degree(ranks, sorted_kw, country, ax)

plt.suptitle('Weighted Degree Rank Distribution', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.savefig('../figures/weighted_degree_rank.png', dpi=150, bbox_inches='tight')
plt.show()

## Degree vs. betweenness centrality

A strong positive correlation indicates a hub and spoke structure where highly
connected airports are also the main transfer points. Outliers with moderate
degree but high betweenness are critical bridge airports.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, country in enumerate(countries):
    ax = axes[idx // 2, idx % 2]
    x, y = degree_betweenness(graphs[country])
    plot_degree_betweenness(x, y, country, ax, log=False)

plt.suptitle('Degree vs Betweenness Centrality (Linear)', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.savefig('../figures/degree_betweenness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, country in enumerate(countries):
    ax = axes[idx // 2, idx % 2]
    x, y = degree_betweenness(graphs[country])
    plot_degree_betweenness(x, y, country, ax, log=True)

plt.suptitle('Degree vs Betweenness Centrality (Log Scale)', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.savefig('../figures/degree_betweenness_loglog.png', dpi=150, bbox_inches='tight')
plt.show()

## Degree assortativity

Negative assortativity (r < 0) means high degree airports connect primarily
to low degree airports, which is common in hub and spoke networks.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, country in enumerate(countries):
    ax = axes[idx // 2, idx % 2]
    r_val, k_vals, knn_vals, sorted_k, avg_knn = assortativity(graphs[country])
    plot_assortativity(k_vals, knn_vals, sorted_k, avg_knn, r_val, country, ax)

plt.suptitle('Degree Assortativity (k vs average neighbour degree)', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.savefig('../figures/assortativity.png', dpi=150, bbox_inches='tight')
plt.show()

## Core identification with the k+ method

The core boundary r* is defined where k+ reaches its maximum. Larger core
sizes indicate higher structural robustness but lower economic efficiency.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, country in enumerate(countries):
    ax = axes[idx // 2, idx % 2]
    ranks, k_plus_curve, peak_rank, peak_val = kplus_core(graphs[country])
    plot_kplus_core(ranks, k_plus_curve, peak_rank, peak_val, country, ax)

plt.suptitle('Core Size via k+ Peak Method', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.savefig('../figures/core_periphery.png', dpi=150, bbox_inches='tight')
plt.show()